In [14]:
from pathlib import Path
import torch

import pickle
from rgfn import ReactionDataFactory, ReactionEnv
base_dir = Path('../../')
reaction_env = ReactionEnv(
    data_factory=ReactionDataFactory(reaction_path=base_dir / 'data/synflow/templates.txt',
                                     fragment_path=base_dir / 'data/synflow/fragment_64k.txt'),
    max_num_reactions=4
)
with open(base_dir / 'trajs.pkl', 'rb') as f:
    trajectories_container = pickle.load(f)
trajectories_container

Using 64000 fragments, 105 reactions, and 197 anchored reactions


TrajectoriesContainer(forward_trajectories=TrajectoriesList(n_trajectories=64):
 S0 =A0(41327)=> SA(F(41327), 0) =AA(149)=> SB(F(41327), R'([OH:3][C$(C(=O)([CX4,c])),C$([CH](=O)):2]=[O:4].[N$([NH2,NH3+1]([CX4,c])),N$([NH]([CX4,c])([CX4,c])):6] >> [N+0:6]-[C:2]=[O:4], 0, 149), 0) =AB(41234)=> SC(F(41327), R'([OH:3][C$(C(=O)([CX4,c])),C$([CH](=O)):2]=[O:4].[N$([NH2,NH3+1]([CX4,c])),N$([NH]([CX4,c])([CX4,c])):6] >> [N+0:6]-[C:2]=[O:4], 0, 149), (F(41234),), 0) =AC(F(41327), R'([OH:3][C$(C(=O)([CX4,c])),C$([CH](=O)):2]=[O:4].[N$([NH2,NH3+1]([CX4,c])),N$([NH]([CX4,c])([CX4,c])):6] >> [N+0:6]-[C:2]=[O:4], 0, 149), (F(41234),), N#Cc1ccc(-c2ocnc2C(=O)Nc2c(Br)cc(I)cc2C#N)cc1)=> SA(N#Cc1ccc(-c2ocnc2C(=O)Nc2c(Br)cc(I)cc2C#N)cc1, 1) =AA(88)=> SB(N#Cc1ccc(-c2ocnc2C(=O)Nc2c(Br)cc(I)cc2C#N)cc1, R'([I][#6;$([#6]~[#6]);!$([#6]([Cl,Br,I])[Cl,Br,I]);!$([#6]=O):3].[#6:1][C:2]#[#7;D1] >> [#6:1][C:2](=O)[#6:3], 1, 88), 1) =AB(40957)=> SC(N#Cc1ccc(-c2ocnc2C(=O)Nc2c(Br)cc(I)cc2C#N)cc1, R'([I][#6;$([#6]~[#6]);

In [15]:
trajectories = trajectories_container.forward_trajectories
backward_states = trajectories.get_non_source_states_flat()  # [n_actions]
forward_states = trajectories.get_non_last_states_flat()  # [n_actions]
backward_action_spaces = trajectories.get_backward_action_spaces_flat()  # [n_actions]
actions = trajectories.get_actions_flat()  # [n_actions]

In [19]:
from rgfn.gfns.reaction_gfn.api.reaction_api import ReactionActionSpace0orCBackward

for action, action_space, forward_state, backward_state in zip(actions, backward_action_spaces, forward_states,
                                                               backward_states):
    if isinstance(action_space, ReactionActionSpace0orCBackward) and action not in action_space.possible_actions:
        next_state = reaction_env.apply_forward_actions([forward_state], [action])[0]
        new_backward_action_space = reaction_env.get_backward_action_spaces([next_state])[0]
        print(new_backward_action_space == action_space)
        print(action in new_backward_action_space.possible_actions)
        # for a in action_space.possible_actions:
        #     print(a)
        # # print('-------------------')
        # # for a in reaction_env.get_backward_action_spaces([backward_state])[0].possible_actions:
        # #     print(a)
        # print('-------------------')
        # print(reaction_env.reversed().get_forward_action_spaces([backward_state])[0].get_idx_of_action(action))
        # raise ValueError

False
True


In [26]:
from rgfn.gfns.reaction_gfn.api.data_structures import Molecule
from rgfn.gfns.reaction_gfn.api.reaction_api import ReactionStateA

ReactionStateA(Molecule('CCC'), 2) == ReactionStateA(Molecule('CCCC'), 2)

False

In [17]:
pickle.load(open(base_dir / 'backward_policy.pkl', 'rb'))

RuntimeError: Attempting to deserialize object on a CUDA device but torch.cuda.is_available() is False. If you are running on a CPU-only machine, please use torch.load with map_location=torch.device('cpu') to map your storages to the CPU.

In [27]:
from dataclasses import dataclass


@dataclass(order=True)
class Point:
    x: int
    y: int

    def __hash__(self):
        # Custom hash logic that might cause a collision
        return hash(self.x)  # Only use x for hashing, ignoring y

p1 = Point(2, 3)
p2 = Point(2, 4)  # Different y, but same hash value due to custom hash

my_dict = {p1: "First", p2: "Second"}
print(my_dict)  # Output: {Point(x=2, y=3): 'Second'}

{Point(x=2, y=3): 'First', Point(x=2, y=4): 'Second'}
